# Colab Training Notebook

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!pip install -q torchmetrics mlflow tqdm

In [11]:
import torch
import shutil, os, glob
import mlflow
import pandas as pd
import sys
from dotenv import load_dotenv
from pathlib import Path

In [12]:
load_dotenv()

True

In [13]:
# Check GPU
if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [14]:
# Set Drive path
DRIVE_PATH = Path(os.environ["DRIVE_PATH"])
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/splits', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/labeled', exist_ok=True)
print("Drive path ready")

Drive path ready


In [15]:
if str(DRIVE_PATH) not in sys.path:
    sys.path.append(str(DRIVE_PATH))

from src.train import train_model
from src.evaluate import evaluate 
from src.alignment_head import train_alignment_head

from src.image_model import AdImageModel
from src.text_encoder import TextEncoder
from src.alignment import compute_alignment
from src.caption_generator import CaptionGenerator
from src.colors import extract_dominant_colors
from src.platform_guides import PLATFORM_GUIDES


In [16]:
# Set MLflow tracking URI to local SQLite database
mlflow.set_tracking_uri(f"sqlite:///{DRIVE_PATH}/mlflow.db")
mlflow.set_experiment("visiomark-image-model")
print("MLflow ready")

MLflow ready


In [17]:
# Check data splits
splits = {}
for split in ['train', 'val', 'test']:
    path = f"{DRIVE_PATH}/data/splits/{split}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        splits[split] = df
        print(f"{split}: {len(df)} samples")
        print(f"content_type distribution: {df['content_type_label'].value_counts().to_dict()}")
        print(f"mood distribution:         {df['mood_label'].value_counts().to_dict()}")
        print()
    else:
        print(f"{split}.csv not found")

train: 1131 samples
content_type distribution: {1: 609, 2: 355, 0: 167}
mood distribution:         {1: 423, 2: 352, 0: 224, 3: 132}

val: 243 samples
content_type distribution: {1: 131, 2: 76, 0: 36}
mood distribution:         {1: 91, 2: 76, 0: 47, 3: 29}

test: 243 samples
content_type distribution: {1: 131, 2: 76, 0: 36}
mood distribution:         {1: 90, 2: 76, 0: 48, 3: 29}



### Phase 1

In [ ]:
print("Starting Phase 1.")
print("=" * 60)

checkpoints_path = train_model(
    phase=1,
    epochs=15,
    batch_size=16,
    lr=1e-3,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=None,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

print("Phase 1 checkpoints saved to: /checkpoints ")

Starting Phase 1.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 192MB/s]


### Phase 2

In [ ]:
print("Starting Phase 2.")
print("=" * 60)

phase1_checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/best_phase1*.pt'))

if not phase1_checkpoints:
    raise FileNotFoundError("No Phase 1 checkpoint found.")

best_phase1 = phase1_checkpoints[-1]

final_checkpoints_path = train_model(
    phase=2,
    epochs=50,
    batch_size=16,
    lr=1e-5,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=best_phase1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

print("Phase 2 checkpoints saved to: /checkpoints ")

## Evaluation

In [ ]:
phase2_checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/best_phase2*.pt'))

In [ ]:
checkpoint_path = phase2_checkpoints[-1] if phase2_checkpoints else None
if checkpoint_path is None:
    raise FileNotFoundError("No Phase 2 checkpoint found for evaluation.")
print("Checkpoint ready for evaluation")

In [ ]:
result = evaluate(
    checkpoint=checkpoint_path,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    test_csv=f"{DRIVE_PATH}/data/splits/test.csv",
    save_path=f"{DRIVE_PATH}/eval/final_results.json",
)
print("Results saved to eval/final_results.json")

### Extract Image Vectors for Alignment Training


In [ ]:
!pip install -q torchmetrics transformers tqdm

#### Image Vectors

In [ ]:
# Extract image vectors from alignment pairs
import os
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).to(DEVICE)
model.classifier = torch.nn.Identity() 
model.eval()

# Load alignment pairs CSV
# Expected columns: image_name, caption
ALIGNMENT_CSV = f'{DRIVE_PATH}/data/alignment_pairs/alignment_pairs.csv'

if not os.path.exists(ALIGNMENT_CSV):
    print("alignment_pairs.csv not found.")
    print("Upload data/alignment_pairs/ folder first.")
else:
    df = pd.read_csv(ALIGNMENT_CSV)
    print(f"Loaded {len(df)} alignment pairs")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    image_vectors = []
    valid_idx = []
    cleaned_captions = []
    
    print("Extracting vectors for manual pairs")

    with torch.no_grad():
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
            caption = str(row.get("caption", "")).strip()
            words = caption.split()
            word_count = len(words)
            
            if word_count < 8:
                continue  
            
            if word_count > 25:
                words = words[:25]
                caption = " ".join(words)
            
            img_path = f'{DRIVE_PATH}/data/raw/{row["image_name"]}'
            try:
                img = Image.open(img_path).convert('RGB')
                img_t = transform(img).unsqueeze(0).to(DEVICE)
                vec = model(img_t).squeeze(0).cpu()
                
                image_vectors.append(vec)
                valid_idx.append(idx)
                cleaned_captions.append(caption)
            except Exception as e:
                print(f"  Skip {row['image_name']}: {e}")


    image_vectors = torch.stack(image_vectors)  # [N, 1280]
    df_valid = df.iloc[valid_idx].reset_index(drop=True)
    
    df_valid["caption"] = cleaned_captions

    # Save
    torch.save(image_vectors, f'{DRIVE_PATH}/checkpoints/alignment_image_vectors.pt')
    df_valid.to_csv(f'{DRIVE_PATH}/data/alignment_pairs/alignment_pairs_valid.csv', index=False)

    print(f"\n {len(image_vectors)} image vectors saved → Drive")
    print(f"   Shape: {image_vectors.shape}")
    print(f"   Filtered out: {len(df) - len(df_valid)} rows")

##### Text Vectors

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
text_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2').to(DEVICE)
text_model.eval()

In [ ]:
text_vectors = []

with torch.no_grad():
    for caption in tqdm(df_valid["caption"].tolist(), desc="Encoding captions"):
        inputs = tokenizer(caption, return_tensors="pt", truncation=True, max_length=128, padding=True).to(DEVICE)
        out = text_model(**inputs)
        attention_mask = inputs["attention_mask"]
        token_embeddings = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        emb = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        emb = torch.nn.functional.normalize(emb, dim=-1).squeeze(0).cpu()
        text_vectors.append(emb)

text_vectors = torch.stack(text_vectors)  # [N, 384]
torch.save(text_vectors, f'{DRIVE_PATH}/checkpoints/alignment_text_vectors.pt')
print(f"Text vectors saved → shape: {text_vectors.shape}")

In [ ]:
img_vectors = torch.load(f'{DRIVE_PATH}/checkpoints/alignment_image_vectors.pt', map_location='cpu',weights_only=True)
txt_vectors = torch.load(f'{DRIVE_PATH}/checkpoints/alignment_text_vectors.pt', map_location='cpu',weights_only=True)

In [ ]:
alignment_head = train_alignment_head(
    image_vectors=img_vectors,
    text_vectors=txt_vectors,
    epochs=40,
    batch_size=16,
    lr=1e-3,
    device='cuda',
    save_path=f'{DRIVE_PATH}/checkpoints/alignment_head.pt',
)

### quick sanity check

In [ ]:
# Test image model
model = AdImageModel(num_content_types=3, num_moods=4, freeze_encoder=False)
model.load_state_dict(torch.load(f'{DRIVE_PATH}/checkpoints/best_phase2.pt', map_location='cpu', weights_only=True))
model.eval()
print("Image model loaded OK")

# Test text encoder
encoder = TextEncoder()
vec = encoder.encode("Premium leather shoes crafted in Italy")
print(f"Text encoder OK — shape: {vec.shape}")

# Test caption generator
gen = CaptionGenerator()
caption = gen.generate("product showcase", "professional")
print(f"Caption generator OK — {caption}")

print("\nAll src modules working")

In [ ]:
# Copy MLflow DB if using local mode
db_source = f"{DRIVE_PATH}/mlflow.db"
db_backup = f"{DRIVE_PATH}/mlflow_backup.db"

if os.path.exists(db_source):
    shutil.copy2(db_source, db_backup)
    print("MLflow database backed up saved.")
else:
    print("MLflow DB not found.")

print("\nALL TRAINING COMPLETE!")